In [ ]:
!pip install pyspark -q

In [13]:
import os
import shutil

In [14]:
os.makedirs("data_lake/bronze", exist_ok=True)

In [15]:
from google.colab import files
files.upload()

Saving sales.csv to sales (1).csv


{'sales (1).csv': b'order_id,customer_id,customer_name,product,category,quantity,unit_price,discount_percent,payment_method,city,state,order_date,order_status\r\n100001,C0256,Keerthi Mishra,Saree,Clothing,2,6710,20,Credit Card,Mumbai,Maharashtra,2026-01-01,Delivered\r\n100002,C0042,Arjun Singh,T-Shirt,Clothing,6,680,5,Debit Card,Chennai,Tamil Nadu,2025-08-03,Delivered\r\n100003,C0348,Keerthi Rao,Hoodie,Clothing,6,3080,12,UPI,Pune,Maharashtra,2025-05-06,Shipped\r\n100004,C0051,Tanvi Sharma,Running Shoes,Sports,6,2260,35,Net Banking,Lucknow,Uttar Pradesh,2026-07-31,Cancelled\r\n100005,C0227,Aarav Kumar,Coffee 500g,Grocery,10,740,12,Debit Card,Vijayawada,Andhra Pradesh,2026-08-30,Delivered\r\n100006,C0117,Aditya Malhotra,Biography,Books,7,650,10,Credit Card,Pune,Maharashtra,2026-05-09,Shipped\r\n100007,C0187,Siddharth Menon,Biography,Books,4,730,30,Cash on Delivery,Hyderabad,Telangana,2025-12-02,Delivered\r\n100008,C0124,Tanvi Chaudhary,Dining Table,Furniture,6,25970,40,Cash on Delivery,D

In [16]:
shutil.copy("sales.csv", "data_lake/bronze/sales.csv")

'data_lake/bronze/sales.csv'

In [18]:

os.makedirs("data_lake/silver", exist_ok=True)

print("Bronze content:", os.listdir("data_lake/bronze"))
print("Silver content:", os.listdir("data_lake/silver"))

Bronze content: ['sales.csv']
Silver content: []


### Start Spark and read the Bronze file

In [19]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [20]:
spark = (SparkSession.builder
         .appName("SalesDataLake")
         .config("spark.sql.ansi.enabled", "false")  # bad dates become null instead of errors
         .getOrCreate())

raw_df = spark.read.csv("data_lake/bronze/sales.csv", header=True, inferSchema=False)

In [21]:
raw_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [22]:
raw_df.show(5, truncate=False)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra|Saree        |Clothing|2       |6710      |20              |Credit Card   |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh   |T-Shirt      |Clothing|6       |680       |5               |Debit Card    |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao   |Hoodie       |Clothing|6       |3080      |12              |UPI           |Pune      |Maharashtra   |2025-05-06|Shipped     |
|100004  |C0051      |

In [23]:
rows_before = raw_df.count()
print("Total records (before preprocessing):", rows_before)

Total records (before preprocessing): 1000


In [24]:
# Exact duplicate rows
dup_rows = rows_before - raw_df.dropDuplicates().count()
print("Exact duplicate rows:", dup_rows)

Exact duplicate rows: 10


In [25]:
# Null / empty values in every column
raw_df.select([
    F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
    for c in raw_df.columns
]).show()

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [26]:
# Text format problems (extra spaces, mixed capital letters)
raw_df.groupBy("category").count().orderBy("category").show(30)

raw_df.groupBy("city").count().orderBy("city").show(30)

extra_space = raw_df.filter(
    (F.col("city") != F.trim(F.col("city"))) |
    (F.col("customer_name") != F.trim(F.col("customer_name")))
).count()
print("Rows with extra spaces in city/customer_name:", extra_space)

+---------------+-----+
|       category|count|
+---------------+-----+
|         Beauty|  136|
|          Books|  128|
|       CLOTHING|    1|
|       Clothing|  108|
|    ELECTRONICS|    1|
|    Electronics|  103|
|      Furniture|  132|
|        Grocery|  136|
|Home Appliances|  115|
|         SPORTS|    1|
|         Sports|  136|
|         beauty|    2|
|         sports|    1|
+---------------+-----+

+-----------+-----+
|       city|count|
+-----------+-----+
|       NULL|    8|
| Hyderabad |    2|
|    Mumbai |    1|
|  Bengaluru|   91|
|    Chennai|   88|
| Coimbatore|   81|
|      DELHI|    2|
|      Delhi|   81|
|  HYDERABAD|    1|
|  Hyderabad|   94|
|     JAIPUR|    1|
|     Jaipur|   84|
|    Kolkata|   81|
|    Lucknow|   90|
|     Mumbai|  106|
|       Pune|   82|
| Vijayawada|  105|
| coimbatore|    1|
|      delhi|    1|
+-----------+-----+

Rows with extra spaces in city/customer_name: 6


In [27]:
# Invalid numbers
print("quantity <= 0       :", raw_df.filter(F.col("quantity").cast("int") <= 0).count())
print("unit_price <= 0     :", raw_df.filter(F.col("unit_price").cast("double") <= 0).count())
print("discount < 0 or >100:", raw_df.filter((F.col("discount_percent").cast("double") < 0) |
                                             (F.col("discount_percent").cast("double") > 100)).count())

quantity <= 0       : 12
unit_price <= 0     : 8
discount < 0 or >100: 8


In [28]:
# Invalid / inconsistent dates
raw_df.filter(~F.col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$")) \
      .select("order_id", "order_date").show()

raw_df.filter(F.col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$") &
              F.to_date("order_date", "yyyy-MM-dd").isNull()) \
      .select("order_id", "order_date").show()

+--------+----------+
|order_id|order_date|
+--------+----------+
|  100097|31/02/2026|
|  100358|2025/07/14|
|  100448|not-a-date|
|  100969|09-15-2026|
+--------+----------+

+--------+----------+
|order_id|order_date|
+--------+----------+
|  100634|2025-02-30|
|  100637|2026-13-05|
+--------+----------+



### Preprocessing  Silver layer


In [29]:
df = raw_df

df = df.dropDuplicates()

for c in df.columns:
  df = df.withColumn(c, F.trim(F.col(c)))

for c in ["customer_name", "category", "city", "state"]:
    df = df.withColumn(c, F.initcap(F.col(c)))

# Missing values
for c in ["customer_name", "payment_method", "city"]:
    df = df.withColumn(c, F.when(F.col(c) == "", None).otherwise(F.col(c)))
df = df.fillna({"customer_name": "Unknown", "payment_method": "Unknown", "city": "Unknown"})

# Convert numeric columns to numbers
df = (df.withColumn("quantity", F.col("quantity").cast("int"))
        .withColumn("unit_price", F.col("unit_price").cast("double"))
        .withColumn("discount_percent", F.col("discount_percent").cast("double")))

In [30]:
df = df.withColumn("quantity", F.when(F.col("quantity") > 0, F.col("quantity")))
df = df.withColumn("unit_price", F.when(F.col("unit_price") > 0, F.col("unit_price")))
df = df.withColumn("discount_percent",
                   F.when((F.col("discount_percent") >= 0) & (F.col("discount_percent") <= 100),
                          F.col("discount_percent")).otherwise(0.0))

overall_qty_median = df.approxQuantile("quantity", [0.5], 0.01)[0]
overall_price_median = df.approxQuantile("unit_price", [0.5], 0.01)[0]

product_window = Window.partitionBy("product")
product_price_median = F.percentile_approx("unit_price", 0.5).over(product_window)

df = df.withColumn("unit_price",
                   F.coalesce(F.col("unit_price"), product_price_median, F.lit(overall_price_median)))
df = df.withColumn("quantity",
                   F.coalesce(F.col("quantity"), F.lit(int(overall_qty_median))))

In [31]:
df = df.withColumn(
    "order_date",
    F.coalesce(
        F.to_date("order_date", "yyyy-MM-dd"),
        F.to_date("order_date", "yyyy/MM/dd"),
        F.to_date("order_date", "dd/MM/yyyy"),
        F.to_date("order_date", "dd-MM-yyyy"),
        F.to_date("order_date", "MM-dd-yyyy"),
    )
)



In [32]:
print("Rows with invalid date set to null:", df.filter(F.col("order_date").isNull()).count())

Rows with invalid date set to null: 4


In [33]:
df = df.orderBy("order_id")

silver_path = "data_lake/silver/preprocessed_sales"
df.coalesce(1).write.mode("overwrite").option("header", True).csv(silver_path)

###Verification

In [39]:
silver_df = spark.read.csv(
    silver_path,
    header=True,
    inferSchema=True
)

print("Silver Data:")
silver_df.show(10, truncate=False)

print("Final Row Count:", silver_df.count())

Silver Data:
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710.0    |20.0            |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh    |T-Shirt              |Clothing |6       |680.0     |5.0             |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao    |Hoodie               |Clothing |6       |3080.0    |12.0            |UPI  

In [34]:
rows_after = df.count()
print("Rows BEFORE preprocessing:", rows_before)
print("Rows AFTER  preprocessing:", rows_after)
print("Rows removed             :", rows_before - rows_after)

Rows BEFORE preprocessing: 1000
Rows AFTER  preprocessing: 990
Rows removed             : 10


In [35]:
silver_df = spark.read.csv(silver_path, header=True, inferSchema=True)
print("\nSilver data (read back)")
silver_df.printSchema()
silver_df.show(10, truncate=False)
print("Silver record count:", silver_df.count())


Silver data (read back)
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+-------

In [36]:
print("Categories now:")
silver_df.groupBy("category").count().show()

Categories now:
+---------------+-----+
|       category|count|
+---------------+-----+
|         Sports|  138|
|        Grocery|  134|
|    Electronics|  104|
|       Clothing|  108|
|          Books|  127|
|Home Appliances|  113|
|      Furniture|  129|
|         Beauty|  137|
+---------------+-----+



In [37]:
print("Bronze folder:", os.listdir("data_lake/bronze"))

Bronze folder: ['sales.csv']


# Summary of preprocessing operations were performed
## 1. Removed exact duplicate rows.
## 2. Trimmed extra spaces in all text columns.
## 3. Standardized capitalization (Title Case) in name, category, city, state.
## 4. Missing customer_name / payment_method / city were filled with "Unknown".
## 5. Invalid quantity (<= 0) and unit_price (<= 0) were replaced with the median value
##    (unit_price: median of the same product).
## 6. Invalid discount_percent (< 0 or > 100) was set to 0.
## 7. Dates in different formats were converted to yyyy-MM-dd; impossible/invalid dates
##    were set to null (rows were kept).
## 8. Saved the result as CSV in data_lake/silver/preprocessed_sales/.